# 🌌 Demo guiada segura: MLflow y la Biblioteca del Templo Jedi

**Modo recomendado:** ejecutar y narrar, sin escribir código en directo.  
**Duración aproximada:** 10–15 minutos  
**Objetivo:** registrar una ejecución real de Machine Learning y comprobar qué guarda MLflow.

En esta demostración utilizaremos:

- el conjunto de datos **Digits**, con imágenes de números escritos a mano;
- un modelo **Random Forest**;
- **MLflow Tracking** para registrar parámetros, métricas, un artefacto y el modelo;
- una segunda ejecución opcional para comparar resultados.

> **Idea que debe quedar clara:** Scikit-learn entrena el modelo. MLflow conserva la historia de cómo se entrenó y qué resultados obtuvo.

---

**Ruta de la píldora:** demo guiada → reto `03_Orden_66_RETO.ipynb` → corrección con `02_Archivos_Jedi_SOLUCION.ipynb`.
**Entorno validado:** Python 3.12, MLflow 3.14.0, scikit-learn 1.9.0, pandas 2.3.3 y matplotlib 3.11.0.


## 🗺️ Mapa de la demostración

Durante el live coding seguiremos este recorrido:

1. Instalamos las librerías.
2. Importamos las herramientas.
3. Cargamos y dividimos los datos.
4. Creamos el experimento: nuestra **campaña Jedi**.
5. Abrimos una ejecución: nuestra **misión Jedi**.
6. Entrenamos el modelo y registramos la información.
7. Consultamos el expediente que ha guardado MLflow.
8. Creamos una segunda misión y comparamos ambas.

### Frase para introducir el código

> “Ahora vamos a comprobar que la analogía de la Biblioteca Jedi no es solamente una historia. Vamos a registrar una misión real y veremos qué conserva MLflow de ella.”

# 1. Preparar el entorno

Google Colab no incluye MLflow de forma predeterminada, por lo que primero instalamos la librería.

### Qué explico mientras se ejecuta

> “Esta celda no forma parte del experimento. Solo prepara el entorno para poder utilizar MLflow. La opción `-q` hace que la instalación muestre menos mensajes en pantalla.”

In [ ]:
# Entorno validado el 15-07-2026. Ejecuta esta celda una sola vez en Colab.
%pip install -q "mlflow==3.14.0" "scikit-learn==1.9.0" "pandas==2.3.3" "matplotlib==3.11.0"

# 2. Importar las herramientas

Importamos MLflow, las funciones de Scikit-learn y las utilidades necesarias para evaluar el modelo.

### Qué explico

> “Aquí todavía no estamos entrenando ni registrando nada. Solo estamos preparando las herramientas que necesitaremos. Importamos `mlflow` y también `mlflow.sklearn`, porque nuestro modelo pertenece a Scikit-learn.”

In [ ]:
# Librerías generales
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# MLflow y su integración con Scikit-learn
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

# Dataset, modelo y métricas
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

print("MLflow:", mlflow.__version__)

# 3. Preparar los datos y decidir la estrategia

Utilizaremos **Digits** porque es un conjunto visual, cercano y rápido de entrenar. Contiene imágenes de números escritos a mano del 0 al 9. Cada imagen mide 8 × 8 píxeles y se transforma en 64 características.

En la analogía:

- los **datos** son la información de la misión;
- `n_estimators`, `max_depth` y `random_state` son parte de la **estrategia**;
- esa estrategia se decide antes de entrenar, por lo que se registrará como **parámetros**.

### Qué explico

> “Nuestro modelo recibirá imágenes de números escritos a mano y tendrá que reconocer qué dígito aparece. Primero preparamos los datos y decidimos cómo será el modelo. Todavía no sabemos si funcionará bien, por eso estos valores son decisiones previas y no métricas.”

In [ ]:
# Cargamos Digits como DataFrame para conservar los nombres de las columnas.
digits = load_digits(as_frame=True)
X = digits.data
y = digits.target

# Separamos datos de entrenamiento y prueba.
# stratify=y mantiene una proporción similar de cada dígito.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Parámetros de la primera misión.
parametros = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42,
}

print("Muestras totales:", len(X))
print("Filas de entrenamiento:", len(X_train))
print("Filas de prueba:", len(X_test))
print("Clases:", list(digits.target_names))
print("Estrategia elegida:", parametros)

# Enseñamos un ejemplo para visualizar el problema.
plt.imshow(digits.images[0], cmap="gray")
plt.title(f"Dígito real: {digits.target[0]}")
plt.axis("off")
plt.show()

# 4. Crear la Biblioteca y la campaña

MLflow necesita saber dónde guardará la información.

- La **Tracking URI** es la dirección de la Biblioteca.
- El **Experiment** agrupa las ejecuciones relacionadas con un mismo objetivo.
- Cada entrenamiento posterior será una **Run** dentro de ese experimento.

En este Colab utilizamos una base SQLite local para los metadatos.

### Qué explico

> “Con `set_tracking_uri` indicamos dónde estará nuestra Biblioteca. Después, con `set_experiment`, abrimos una campaña llamada `Biblioteca_Templo_Jedi_Digits`. Esta línea no entrena nada; solamente organiza dónde quedarán registradas las misiones.”

In [ ]:
# Base SQLite portátil: funciona tanto en Colab como en Jupyter local.
TRACKING_DB = Path("mlflow_jedi_digits.db").resolve()
TRACKING_URI = f"sqlite:///{TRACKING_DB.as_posix()}"
NOMBRE_EXPERIMENTO = "Biblioteca_Templo_Jedi_Digits_Live"

mlflow.set_tracking_uri(TRACKING_URI)
experimento = mlflow.set_experiment(NOMBRE_EXPERIMENTO)

print("Experimento:", experimento.name)
print("ID del experimento:", experimento.experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())

# 5. Abrir una misión, entrenar y registrar

Esta es la celda principal del live coding.

Dentro de `mlflow.start_run()`:

1. creamos el modelo;
2. lo entrenamos;
3. calculamos las métricas;
4. registramos parámetros y métricas;
5. guardamos una matriz de confusión como artefacto;
6. guardamos el modelo entrenado.

### Diferencias que conviene repetir

- **Parameter:** decisión conocida antes del entrenamiento.
- **Metric:** resultado numérico calculado después.
- **Artifact:** archivo generado por la ejecución.
- **Model:** objeto entrenado que podremos reutilizar.

### Frase antes de ejecutar

> “Con `start_run` abrimos el expediente de una misión. Todo lo que registremos dentro de este bloque quedará asociado a la misma ejecución.”

In [ ]:
# Abrimos una ejecución de MLflow.
with mlflow.start_run(run_name="Mision_Coruscant") as run:

    # 1. Creamos y entrenamos el modelo con la estrategia elegida.
    modelo = RandomForestClassifier(**parametros)
    modelo.fit(X_train, y_train)

    # 2. Realizamos predicciones con los datos que el modelo no vio al entrenar.
    predicciones = modelo.predict(X_test)

    # 3. Calculamos resultados numéricos: las métricas.
    metricas = {
        "accuracy": accuracy_score(y_test, predicciones),
        "precision_macro": precision_score(
            y_test, predicciones, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, predicciones, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, predicciones, average="macro", zero_division=0
        ),
    }

    # 4. Registramos la estrategia y los resultados.
    mlflow.log_params(parametros)
    mlflow.log_metrics(metricas)

    # 5. Añadimos etiquetas para dar contexto a la misión.
    mlflow.set_tags({
        "dataset": "Digits",
        "modelo": "RandomForestClassifier",
        "tipo_demo": "live_coding",
        "universo": "Biblioteca Jedi",
    })

    # 6. Generamos y guardamos un archivo: la matriz de confusión.
    ruta_matriz = Path("matriz_confusion_coruscant.png").resolve()

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predicciones,
        display_labels=digits.target_names,
        cmap="Blues",
    )
    plt.title("Matriz de confusión · Misión Coruscant")
    plt.tight_layout()
    plt.savefig(ruta_matriz, dpi=150)
    plt.show()

    # Registramos el archivo como artefacto.
    mlflow.log_artifact(str(ruta_matriz), artifact_path="evaluacion")

    # 7. Inferimos la firma: qué datos entran y qué predicciones salen.
    firma = infer_signature(X_train, modelo.predict(X_train))

    # 8. Guardamos el modelo completo en formato MLflow.
    informacion_modelo = mlflow.sklearn.log_model(
        sk_model=modelo,
        name="modelo_jedi",
        signature=firma,
        input_example=X_train.head(3),
    )

    # Guardamos el identificador para consultarlo después.
    run_id = run.info.run_id

print("✅ Misión registrada correctamente")
print("Run ID:", run_id)
print("Métricas:", metricas)

# 6. Abrir el expediente que ha creado MLflow

No necesitamos la interfaz gráfica para comprobar el registro. Podemos recuperar la ejecución desde Python.

### Qué explico

> “El entrenamiento ha terminado, pero la información no ha desaparecido. MLflow conserva el nombre de la misión, su estado, los parámetros, las métricas, las etiquetas y los archivos que se han generado.”

In [ ]:
# Recuperamos la ejecución mediante su identificador.
expediente = mlflow.get_run(run_id)

print("Nombre de la misión:", expediente.data.tags.get("mlflow.runName"))
print("Estado:", expediente.info.status)

print("\nPARÁMETROS")
for nombre, valor in expediente.data.params.items():
    print(f"  {nombre}: {valor}")

print("\nMÉTRICAS")
for nombre, valor in expediente.data.metrics.items():
    print(f"  {nombre}: {valor:.4f}")

# Consultamos los artefactos guardados.
cliente = MlflowClient()
artefactos = cliente.list_artifacts(run_id)

print("\nARTEFACTOS REGISTRADOS (el modelo se consulta en Logged Models en MLflow 3)")
for artefacto in artefactos:
    tipo = "carpeta" if artefacto.is_dir else "archivo"
    print(f"  {artefacto.path} ({tipo})")

# 7. Segunda misión opcional

Esta parte puede estar escrita de antemano. Solo hay que ejecutarla si queda tiempo.

Cambiaremos la estrategia para demostrar que un **Experiment** puede contener varias **Runs**. En esta segunda misión utilizamos menos árboles y una profundidad algo menor.

### Qué explico

> “No estamos sustituyendo la primera misión. Estamos creando otra ejecución dentro de la misma campaña. MLflow conservará ambas para que podamos compararlas.”

In [ ]:
# Estrategia diferente para la segunda misión.
parametros_tatooine = {
    "n_estimators": 30,
    "max_depth": 2,
    "random_state": 42,
}

with mlflow.start_run(run_name="Mision_Tatooine") as segunda_run:

    modelo_tatooine = RandomForestClassifier(**parametros_tatooine)
    modelo_tatooine.fit(X_train, y_train)

    pred_tatooine = modelo_tatooine.predict(X_test)

    metricas_tatooine = {
        "accuracy": accuracy_score(y_test, pred_tatooine),
        "precision_macro": precision_score(
            y_test, pred_tatooine, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, pred_tatooine, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, pred_tatooine, average="macro", zero_division=0
        ),
    }

    # Registramos parámetros, métricas y etiquetas.
    mlflow.log_params(parametros_tatooine)
    mlflow.log_metrics(metricas_tatooine)
    mlflow.set_tags({
        "dataset": "Digits",
        "modelo": "RandomForestClassifier",
        "tipo_demo": "comparacion",
        "universo": "Biblioteca Jedi",
    })

print("✅ Segunda misión registrada")
print("Métricas Tatooine:", metricas_tatooine)

# 8. Comparar las misiones

`mlflow.search_runs()` recupera las ejecuciones de un experimento y devuelve una tabla.

### Qué explico

> “Aquí aparece el verdadero valor de MLflow. Ya no tenemos dos modelos sueltos ni dos resultados apuntados en lugares distintos. Tenemos dos expedientes comparables, con sus decisiones y sus resultados en la misma tabla.”

In [ ]:
# Recuperamos todas las runs del experimento y las ordenamos por accuracy.
comparacion = mlflow.search_runs(
    experiment_names=[NOMBRE_EXPERIMENTO],
    order_by=["metrics.accuracy DESC"],
)

# Seleccionamos las columnas más útiles para la demostración.
columnas = [
    "tags.mlflow.runName",
    "params.n_estimators",
    "params.max_depth",
    "metrics.accuracy",
    "metrics.precision_macro",
    "metrics.recall_macro",
    "metrics.f1_macro",
    "status",
]

comparacion[columnas].rename(columns={
    "tags.mlflow.runName": "mision",
    "params.n_estimators": "n_estimators",
    "params.max_depth": "max_depth",
    "metrics.accuracy": "accuracy",
    "metrics.precision_macro": "precision",
    "metrics.recall_macro": "recall",
    "metrics.f1_macro": "f1",
})

# 9. Cierre del live coding

## Lo que acabamos de hacer

| Biblioteca Jedi | MLflow | En el código |
|---|---|---|
| Biblioteca | Tracking URI | `mlflow.set_tracking_uri()` |
| Campaña | Experiment | `mlflow.set_experiment()` |
| Misión | Run | `mlflow.start_run()` |
| Estrategia | Parameters | `mlflow.log_params()` |
| Informe | Metrics | `mlflow.log_metrics()` |
| Holocrón | Artifact | `mlflow.log_artifact()` |
| Caballero entrenado | Model | `mlflow.sklearn.log_model()` |

### Cierre oral

> “Scikit-learn ha entrenado el modelo. MLflow no ha mejorado su accuracy ni ha decidido los parámetros. Lo que ha hecho es conservar la historia completa: qué probamos, qué resultado obtuvimos, qué archivo generamos y qué modelo quedó entrenado. Esa es la función de la Biblioteca del Templo Jedi: que ninguna misión y ningún aprendizaje se pierdan.”

---

## 🛟 Plan B para la exposición

- Usa este notebook como **demo guiada**: todo el código está escrito; tú solo ejecutas y explicas.
- Ejecuta la instalación antes de compartir pantalla y conserva una pestaña con las salidas ya generadas.
- Si una celda falla, no improvises código: muestra la captura de respaldo y continúa con la explicación conceptual.
- En Colab, valida las runs con `mlflow.search_runs()`; `localhost:5000` solo es accesible directamente en Jupyter local.
- Si se vuelve a ejecutar una celda de entrenamiento, MLflow creará otra run: es comportamiento esperado, no un error.
